In [1]:
##### Converts raw raster on field size into final varaiable needed 
# pixel and subnational (vector) scale
# variables: % of area with very large / large / medium / small  farms

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.features import geometry_mask

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# raw raster
raw = f"{cd}/Data/Raw/Predictors/Field_size_Lesiv/dominant_field_size_categories.tif"

# Save paths
pixels_vlarge = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_vlarge.tif"
pixels_large = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_large.tif"
pixels_medium = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_medium.tif"
pixels_small = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_small.tif"
pixels_vsmall = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_vsmall.tif"

vectors = f"{cd}/Data/Clean/Predictors/Vectors/field_size_shares.csv"
vectors_country = f"{cd}/Data/Clean/Predictors/Vectors/Country_averages/field_size_shares.csv"

In [3]:
#### Run resampling for pixel scale 

### PREP 

# set list of field size categories 
categories = {
    "vlarge": (3502, pixels_vlarge),
    "large":  (3503, pixels_large),
    "medium": (3504, pixels_medium),
    "small":  (3505, pixels_small),
    "vsmall": (3506, pixels_vsmall),
}

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)

# reproject countries once
if countries.crs != dst_crs:
    countries = countries.to_crs(dst_crs)
countries = countries.reset_index(drop=True)


### RESAMPLE AND GAP-FILL
with rasterio.open(raw) as src:
    data          = src.read(1)
    src_nodata    = src.nodata
    src_transform = src.transform
    src_crs       = src.crs

    # mask out both nodata AND no-field (0) so shares are just of areas with fields
    exclude_mask = (data == 0)
    if src_nodata is not None:
        exclude_mask |= (data == src_nodata)

    for name, (code, out_path) in categories.items():

        # 1.0 where this category, 0.0 for other field categories, nan for no-field/nodata
        binary = np.where(exclude_mask, np.nan, (data == code).astype(np.float32))

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=binary,
            destination=dst_array,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.average,
            src_nodata=np.nan,
            dst_nodata=np.nan,
        )

        # align to reference mask
        dst_array[~ref_valid] = np.nan

        needs_fill = ref_valid & np.isnan(dst_array)
        print(f"[{name}] Pixels needing fill: {needs_fill.sum()}")

        if needs_fill.any():

            country_stats = rasterstats.zonal_stats(
                countries,
                dst_array,
                affine=dst_transform,
                stats=["mean"],
                nodata=np.nan,
            )

            country_means = {
                i: s["mean"] for i, s in enumerate(country_stats)
                if s["mean"] is not None
            }

            fill_array = np.full(dst_shape, np.nan, dtype=np.float32)

            for idx, row in countries.iterrows():
                mean_val = country_means.get(idx)
                if mean_val is None:
                    continue

                country_mask = geometry_mask(
                    [row.geometry],
                    transform=dst_transform,
                    invert=True,
                    out_shape=dst_shape,
                )

                fill_array[needs_fill & country_mask] = mean_val

            dst_array = np.where(needs_fill, fill_array, dst_array).astype(np.float32)

            still_missing = ref_valid & np.isnan(dst_array)
            if still_missing.any():
                print(f"  [{name}] Warning: {still_missing.sum()} pixels still unfilled. "
                      f"These will be saved as NoData.")

        # save
        out_meta = dst_meta.copy()
        out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

        out_arr = dst_array.copy()
        out_arr[np.isnan(out_arr)] = -9999

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(out_arr, 1)

        print(f"  [{name}] Saved → {out_path}")

[vlarge] Pixels needing fill: 1361035
  [vlarge] Warning: 4391 pixels still unfilled. These will be saved as NoData.
  [vlarge] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_vlarge.tif
[large] Pixels needing fill: 1361035
  [large] Warning: 4391 pixels still unfilled. These will be saved as NoData.
  [large] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_large.tif
[medium] Pixels needing fill: 1361035
  [medium] Warning: 4391 pixels still unfilled. These will be saved as NoData.
  [medium] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_medium.tif
[small] Pixels needing fill: 1361035
  [small] Warning: 4391 pixels still unfilled. These will be saved as NoData.
  [small] Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_small.tif
[vsmall] Pixels need

In [4]:
#### Run resampling for vector scale (sub-national)

### Set-up 

# repoject GDF to match raster 
with rasterio.open(raw) as src:
    raster_crs = src.crs
    data = src.read(1)
    src_transform = src.transform
    src_nodata = src.nodata

gdf_proj = total_geo.to_crs(raster_crs)

# mask out both nodata and no-field (0) pixels by setting them to a single nodata value
clean = data.copy().astype(np.float32)
clean[(data == 0)] = np.nan
if src_nodata is not None:
    clean[(data == src_nodata)] = np.nan
clean[(data == -9999)] = np.nan


### Run zonal stats
# get pixel counts per category per polygon
zonal = rasterstats.zonal_stats(
    gdf_proj,
    clean,
    affine=src_transform,
    categorical=True,
    nodata=np.nan
)

### Convert to df

# map raster codes to column names
code_map = {
    3502: "share_vlarge_field",
    3503: "share_large_field",
    3504: "share_medium_field",
    3505: "share_small_field",
    3506: "share_vsmall_field",
}

# get project ID's
result = total_geo[["PROJ_ID"]].copy()

# add in zonal stats
for code, col_name in code_map.items():
    counts = [z.get(code, 0) for z in zonal]
    totals = [sum(z.get(c, 0) for c in code_map.keys()) for z in zonal]
    result[col_name] = [
        c / t if t > 0 else None
        for c, t in zip(counts, totals)
    ]

### Save
result.to_csv(vectors, index=False)

In [ ]:
#### Run resampling for vector scale (country averages)

### Set-up 

# repoject GDF to match raster 
with rasterio.open(raw) as src:
    raster_crs = src.crs
    data = src.read(1)
    src_transform = src.transform
    src_nodata = src.nodata

gdf_proj = countries.to_crs(raster_crs)

# mask out both nodata and no-field (0) pixels by setting them to a single nodata value
clean = data.copy().astype(np.float32)
clean[(data == 0)] = np.nan
if src_nodata is not None:
    clean[(data == src_nodata)] = np.nan
clean[(data == -9999)] = np.nan


### Run zonal stats
# get pixel counts per category per polygon
zonal = rasterstats.zonal_stats(
    gdf_proj,
    clean,
    affine=src_transform,
    categorical=True,
    nodata=np.nan
)

### Convert to df

# map raster codes to column names
code_map = {
    3502: "share_vlarge_field",
    3503: "share_large_field",
    3504: "share_medium_field",
    3505: "share_small_field",
    3506: "share_vsmall_field",
}

# get project ID's
result = countries[["GID_0"]].copy()

# add in zonal stats
for code, col_name in code_map.items():
    counts = [z.get(code, 0) for z in zonal]
    totals = [sum(z.get(c, 0) for c in code_map.keys()) for z in zonal]
    result[col_name] = [
        c / t if t > 0 else None
        for c, t in zip(counts, totals)
    ]

### Save
result.to_csv(vectors_country, index=False)